In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

pd.set_option("display.max_columns", None)

In [2]:
train_path = "../data/KDDTrain+.txt"
test_path = "../data/KDDTest+.txt"

train = pd.read_csv(train_path, header=None)
test = pd.read_csv(test_path, header=None)

print("Train shape:", train.shape)
print("Test shape :", test.shape)

Train shape: (125973, 43)
Test shape : (22544, 43)


In [3]:
columns = [
    "duration",
    "protocol_type",
    "service",
    "flag",
    "src_bytes",
    "dst_bytes",
    "land",
    "wrong_fragment",
    "urgent",
    "hot",
    "num_failed_logins",
    "logged_in",
    "num_compromised",
    "root_shell",
    "su_attempted",
    "num_root",
    "num_file_creations",
    "num_shells",
    "num_access_files",
    "num_outbound_cmds",
    "is_host_login",
    "is_guest_login",
    "count",
    "srv_count",
    "serror_rate",
    "srv_serror_rate",
    "rerror_rate",
    "srv_rerror_rate",
    "same_srv_rate",
    "diff_srv_rate",
    "srv_diff_host_rate",
    "dst_host_count",
    "dst_host_srv_count",
    "dst_host_same_srv_rate",
    "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate",
    "dst_host_srv_serror_rate",
    "dst_host_rerror_rate",
    "dst_host_srv_rerror_rate",
    "label",
    "difficulty"
]

train.columns = columns
test.columns = columns

train.head()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,root_shell,su_attempted,num_root,num_file_creations,num_shells,num_access_files,num_outbound_cmds,is_host_login,is_guest_login,count,srv_count,serror_rate,srv_serror_rate,rerror_rate,srv_rerror_rate,same_srv_rate,diff_srv_rate,srv_diff_host_rate,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,2,0.0,0.0,0.0,0.0,1.00,0.00,0.00,150,25,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,13,1,0.0,0.0,0.0,0.0,0.08,0.15,0.00,255,1,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,123,6,1.0,1.0,0.0,0.0,0.05,0.07,0.00,255,26,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,5,5,0.2,0.2,0.0,0.0,1.00,0.00,0.00,30,255,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,30,32,0.0,0.0,0.0,0.0,1.00,0.00,0.09,255,255,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21


In [4]:
train_attacks = set(
    train.loc[train["label"] != "normal", "label"].unique()
)

print("Number of attack types in Train:", len(train_attacks))
print(sorted(train_attacks))

Number of attack types in Train: 22
['back', 'buffer_overflow', 'ftp_write', 'guess_passwd', 'imap', 'ipsweep', 'land', 'loadmodule', 'multihop', 'neptune', 'nmap', 'perl', 'phf', 'pod', 'portsweep', 'rootkit', 'satan', 'smurf', 'spy', 'teardrop', 'warezclient', 'warezmaster']


In [5]:
test_attacks = set(
    test.loc[test["label"] != "normal", "label"].unique()
)

print("Number of attack types in Test:", len(test_attacks))
print(sorted(test_attacks))

Number of attack types in Test: 37
['apache2', 'back', 'buffer_overflow', 'ftp_write', 'guess_passwd', 'httptunnel', 'imap', 'ipsweep', 'land', 'loadmodule', 'mailbomb', 'mscan', 'multihop', 'named', 'neptune', 'nmap', 'perl', 'phf', 'pod', 'portsweep', 'processtable', 'ps', 'rootkit', 'saint', 'satan', 'sendmail', 'smurf', 'snmpgetattack', 'snmpguess', 'sqlattack', 'teardrop', 'udpstorm', 'warezmaster', 'worm', 'xlock', 'xsnoop', 'xterm']


In [6]:
unseen_attacks = sorted(test_attacks - train_attacks)

print("Unseen / Zero-Day attack types:")
print(unseen_attacks)

print("\nNumber of unseen attack types:", len(unseen_attacks))

Unseen / Zero-Day attack types:
['apache2', 'httptunnel', 'mailbomb', 'mscan', 'named', 'processtable', 'ps', 'saint', 'sendmail', 'snmpgetattack', 'snmpguess', 'sqlattack', 'udpstorm', 'worm', 'xlock', 'xsnoop', 'xterm']

Number of unseen attack types: 17


In [7]:
zero_day_counts = (
    test[test["label"].isin(unseen_attacks)]["label"]
    .value_counts()
    .sort_index()
)

print(zero_day_counts)

print("\nTotal zero-day samples:", zero_day_counts.sum())

label
apache2          737
httptunnel       133
mailbomb         293
mscan            996
named             17
processtable     685
ps                15
saint            319
sendmail          14
snmpgetattack    178
snmpguess        331
sqlattack          2
udpstorm           2
worm               2
xlock              9
xsnoop             4
xterm             13
Name: count, dtype: int64

Total zero-day samples: 3750


In [8]:
seen_mask = ~test["label"].isin(unseen_attacks)

test_seen = test[seen_mask].copy()

In [9]:
zero_day_mask = test["label"].isin(unseen_attacks)

test_zero_day = test[zero_day_mask].copy()

In [10]:
print("Test Seen shape    :", test_seen.shape)
print("Test Zero-Day shape:", test_zero_day.shape)

Test Seen shape    : (18794, 43)
Test Zero-Day shape: (3750, 43)


In [11]:
train["binary_label"] = (train["label"] != "normal").astype(int)

test_seen["binary_label"] = (
    test_seen["label"] != "normal"
).astype(int)

test_zero_day["binary_label"] = (
    test_zero_day["label"] != "normal"
).astype(int)

In [12]:
print(train["binary_label"].value_counts())
print()
print(test_seen["binary_label"].value_counts())
print()
print(test_zero_day["binary_label"].value_counts())

binary_label
0    67343
1    58630
Name: count, dtype: int64

binary_label
0    9711
1    9083
Name: count, dtype: int64

binary_label
1    3750
Name: count, dtype: int64


In [13]:
feature_columns = [
    col for col in columns
    if col not in ["label", "difficulty"]
]

In [14]:
X_train_raw = train[feature_columns].copy()
y_train_zero_day = train["binary_label"].copy()

X_test_seen_raw = test_seen[feature_columns].copy()
y_test_seen = test_seen["binary_label"].copy()

X_test_zero_day_raw = test_zero_day[feature_columns].copy()
y_test_zero_day = test_zero_day["binary_label"].copy()

In [15]:
categorical_cols = [
    "protocol_type",
    "service",
    "flag"
]

numeric_cols = [
    col for col in feature_columns
    if col not in categorical_cols
]

print("Categorical features:")
print(categorical_cols)

print("\nNumber of numerical features:", len(numeric_cols))

Categorical features:
['protocol_type', 'service', 'flag']

Number of numerical features: 38


In [16]:
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

In [17]:
X_train_cat = encoder.fit_transform(
    X_train_raw[categorical_cols]
)

X_test_seen_cat = encoder.transform(
    X_test_seen_raw[categorical_cols]
)

X_test_zero_day_cat = encoder.transform(
    X_test_zero_day_raw[categorical_cols]
)

In [18]:
X_train_num = X_train_raw[numeric_cols].values
X_test_seen_num = X_test_seen_raw[numeric_cols].values
X_test_zero_day_num = X_test_zero_day_raw[numeric_cols].values

In [19]:
X_train_zero_day = np.hstack([
    X_train_num,
    X_train_cat
])

X_test_seen = np.hstack([
    X_test_seen_num,
    X_test_seen_cat
])

X_test_zero_day = np.hstack([
    X_test_zero_day_num,
    X_test_zero_day_cat
])

In [20]:
print("X_train_zero_day :", X_train_zero_day.shape)
print("X_test_seen      :", X_test_seen.shape)
print("X_test_zero_day  :", X_test_zero_day.shape)

print()

print("y_train_zero_day :", y_train_zero_day.shape)
print("y_test_seen      :", y_test_seen.shape)
print("y_test_zero_day  :", y_test_zero_day.shape)

X_train_zero_day : (125973, 122)
X_test_seen      : (18794, 122)
X_test_zero_day  : (3750, 122)

y_train_zero_day : (125973,)
y_test_seen      : (18794,)
y_test_zero_day  : (3750,)


In [21]:
scaler = StandardScaler()

X_train_zero_day_scaled = scaler.fit_transform(
    X_train_zero_day
)

X_test_seen_scaled = scaler.transform(
    X_test_seen
)

X_test_zero_day_scaled = scaler.transform(
    X_test_zero_day
)

In [22]:
scaler = StandardScaler()

X_train_zero_day_scaled = scaler.fit_transform(
    X_train_zero_day
)

X_test_seen_scaled = scaler.transform(
    X_test_seen
)

X_test_zero_day_scaled = scaler.transform(
    X_test_zero_day
)

In [23]:
print("Train:")
print("Normal:", (y_train_zero_day == 0).sum())
print("Attack:", (y_train_zero_day == 1).sum())

print("\nTest Seen:")
print("Normal:", (y_test_seen == 0).sum())
print("Attack:", (y_test_seen == 1).sum())

print("\nTest Zero-Day:")
print("Normal:", (y_test_zero_day == 0).sum())
print("Attack:", (y_test_zero_day == 1).sum())

Train:
Normal: 67343
Attack: 58630

Test Seen:
Normal: 9711
Attack: 9083

Test Zero-Day:
Normal: 0
Attack: 3750


In [24]:
print("\nZero-Day attack distribution:")
print(test_zero_day["label"].value_counts().sort_index())


Zero-Day attack distribution:
label
apache2          737
httptunnel       133
mailbomb         293
mscan            996
named             17
processtable     685
ps                15
saint            319
sendmail          14
snmpgetattack    178
snmpguess        331
sqlattack          2
udpstorm           2
worm               2
xlock              9
xsnoop             4
xterm             13
Name: count, dtype: int64


In [25]:
assert set(unseen_attacks).isdisjoint(train_attacks)

assert set(test_zero_day["label"]).issubset(
    set(unseen_attacks)
)

assert set(test_seen["label"]).isdisjoint(
    set(unseen_attacks)
)

print("✓ Zero-Day split is correct.")
print("✓ No unseen attack type exists in training.")
print("✓ Test Seen contains no Zero-Day attack type.")

✓ Zero-Day split is correct.
✓ No unseen attack type exists in training.
✓ Test Seen contains no Zero-Day attack type.


In [26]:
joblib.dump(
    X_train_zero_day,
    "../data/X_train_zero_day.pkl"
)

joblib.dump(
    y_train_zero_day,
    "../data/y_train_zero_day.pkl"
)

joblib.dump(
    X_test_seen,
    "../data/X_test_seen.pkl"
)

joblib.dump(
    y_test_seen,
    "../data/y_test_seen.pkl"
)

joblib.dump(
    X_test_zero_day,
    "../data/X_test_zero_day.pkl"
)

joblib.dump(
    y_test_zero_day,
    "../data/y_test_zero_day.pkl"
)

['../data/y_test_zero_day.pkl']

In [27]:
joblib.dump(
    X_train_zero_day_scaled,
    "../data/X_train_zero_day_scaled.pkl"
)

joblib.dump(
    X_test_seen_scaled,
    "../data/X_test_seen_scaled.pkl"
)

joblib.dump(
    X_test_zero_day_scaled,
    "../data/X_test_zero_day_scaled.pkl"
)

['../data/X_test_zero_day_scaled.pkl']

In [28]:
joblib.dump(
    encoder,
    "../data/zero_day_encoder.pkl"
)

joblib.dump(
    scaler,
    "../data/zero_day_scaler.pkl"
)

['../data/zero_day_scaler.pkl']